SEs for the country groupings we are using in the 2-pager visual

In [6]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import matplotlib.pyplot as plt

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [2]:
load_dotenv()
DATA_DIR = os.environ["DATA_DIR"]
# EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]

EFFECTS_URI = os.path.join(DATA_DIR, "260828_effects.zarr") # Local version 
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]

In [3]:
# Projection Effects
effect = xr.open_datatree(
    EFFECTS_URI,
    engine="zarr",
    chunks={})

baseline_period = analysis_utils.get_baseline_period(effect, years=30)
# Impact Regions
_polygons = (
    gpd.read_parquet(os.path.join(DATA_DIR, IMPACT_REGION_POLYGONS))
    .rename(columns={"hierid": "region"})
    .set_index("region")
    .set_crs(epsg=4326)  # Assuming the data is WGS-82.
)

# Socioeconomics
socioeconomics = xr.open_zarr(os.path.join(DATA_DIR, SOCIOECONOMICS_URI))
socioeconomics = socioeconomics.sel(year=2026)[
    ["pop0to4", "pop5to64", "pop65plus", "pop", "gdppc", "iso3"]
]

In [30]:
dims = ['number', 'sample']
months = [1, 8, 9, 10, 11, 12]
impact = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in dims}),
    socioeconomics,
    ensemble=True,
    baseline_period=baseline_period,
    hotonly="hotonly",
    rate=False,
    age_weight=True,
)
impact = impact.sel(month = months)

In [31]:
impact, polygon = analysis_utils.aggregate_by_iso(impact, _polygons, operation="sum")

In [32]:
#Sahel (Sudan+Nigeria+Niger+Chad)
impact_sahel = impact.sel(ISO = ["NGA", "SDN", "NER", "TCD"]).sum(dim = 'ISO')
impact_sahel = analysis_utils.compute_stats(impact_sahel, dim=["number", 'sample'])

In [33]:
impact_sahel.to_dataframe().reset_index().to_csv("hotonly_impact_sahel.csv", index=False)

In [34]:
#SE Asia (Vietnam, Philippines, Thailand, Cambodia)
impact_seasia = impact.sel(ISO = ["VNM", "PHL", "THA", "KHM"]).sum(dim = 'ISO')
impact_seasia = analysis_utils.compute_stats(impact_seasia, dim=["number", 'sample'])
impact_seasia.to_dataframe().reset_index().to_csv("hotonly_impact_seasia.csv", index=False)